In [2]:
import pandas as pd

/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [7]:
def normalize_to_quarterly(df, date_col, value_cols, filename):

    # value_cols as a list
    if isinstance(value_cols, str):
        value_cols = [value_cols]

    #convert date column to datetime
    try:
        df["Date"] = pd.to_datetime(df[date_col])
    except:
        df["Date"] = pd.to_datetime(df[date_col], format="%b %Y", errors="coerce")

    df = df.dropna(subset=["Date"])

    # cronologically ordered
    df = df.sort_values("Date")

    # take numerical columns
    available_cols = [c for c in value_cols if c in df.columns]

    if not available_cols:
        raise ValueError("Nessuna delle colonne specificate è presente nel dataframe.")

    # Resample to quarterly --> mean values
    df_quarter = (
        df.set_index("Date")[available_cols]
        .resample("Q")
        .mean()
        .reset_index()
    )

    # Change name Date → QuarterEnd
    df_quarter = df_quarter.rename(columns={"Date": "QuarterEnd"})

    # Order descending by date
    df_quarter = df_quarter.sort_values("QuarterEnd", ascending=False)

    # Save to csv
    output_path = f"quarterly_data/{filename}_quarterly.csv"
    df_quarter.to_csv(output_path, index=False)



In [47]:
#for daily data

def normalize_daily_to_quarterly(df, date_col, value_cols, filename):

    # value_cols as a list
    if isinstance(value_cols, str):
        value_cols = [value_cols]

    #convert date column to datetime
    df["Date"] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=["Date"])

    # cronologically ordered
    df = df.sort_values("Date")

    # take numerical columns
    available_cols = [c for c in value_cols if c in df.columns]
    if not available_cols:
        raise ValueError("Nessuna colonna valida trovata nel dataset.")

    # Resample to monthly --> mean values
    df_monthly = (
        df.set_index("Date")[available_cols]
        .resample("M")
        .mean()
    )

    # resample to quarterly
    df_quarterly = df_monthly.resample("Q").mean().reset_index()

    # Change name Date → QuarterEnd
    df_quarterly = df_quarterly.rename(columns={"Date": "QuarterEnd"})

    # Order descending by date
    df_quarterly = df_quarterly.sort_values("QuarterEnd", ascending=False)

    # Save to csv
    output_path = f"quarterly_data/{filename}_quarterly.csv"
    df_quarterly.to_csv(output_path, index=False)



In [48]:
#if year and month are distinct columns 

def quarterly_from_year_month(df, year_col, month_col, value_col, filename):

    df = df.copy()

    # clean months
    df[month_col] = pd.to_numeric(df[month_col], errors='coerce').astype("Int64")
    df[year_col] = pd.to_numeric(df[year_col], errors='coerce').astype("Int64")

    #convert date column to datetime
    df["Date"] = pd.to_datetime(
        df[year_col].astype(str) + "-" + df[month_col].astype(str) + "-01",
        errors="coerce"
    )

    df = df.dropna(subset=["Date"])

    # sort
    df = df.sort_values("Date").set_index("Date")

    # quarterly resample
    quarterly = df[[value_col]].resample("Q").mean().reset_index()

    quarterly = quarterly.rename(columns={"Date": "QuarterEnd"})
    
    quarterly = quarterly.sort_values("QuarterEnd", ascending=False)

    quarterly.to_csv(f"quarterly_data/{filename}_quartely.csv", index=False)


In [49]:
#for natural disasters

def disasters_to_quarterly(df, begin_col, cost_cols, filename):

    # cost_col as a list
    if isinstance(cost_cols, str):
        cost_cols = [cost_cols]

    #convert date column to datetime
    df["BeginDate"] = pd.to_datetime(df[begin_col], errors="coerce")
    df = df.dropna(subset=["BeginDate"])

    # cronologically ordered
    df = df.sort_values("BeginDate")

    # create quarter column: es. "2024Q1"
    df["Quarter"] = df["BeginDate"].dt.to_period("Q").astype(str)

    # Sum the costs per quarter
    df_quarter = df.groupby("Quarter")[cost_cols].sum().reset_index()

    # create a real date
    df_quarter["QuarterEnd"] = pd.PeriodIndex(df_quarter["Quarter"], freq="Q").end_time
    df_quarter["QuarterEnd"] = df_quarter["QuarterEnd"].dt.date


    # Order descending by date
    df_quarter = df_quarter.sort_values("QuarterEnd", ascending=False)

    #save as csv
    output_path = f"quarterly_data/{filename}_quarterly.csv"
    df_quarter.to_csv(output_path, index=False)


In [50]:
def resample_quarterly_nucor(df, year_col="Year", quarter_col="Quarter", value_cols=None, filename="nucor"):
    
    df = df.copy()
    
    if value_cols is None:
        value_cols = df.select_dtypes(include="number").columns.tolist()
        value_cols = [col for col in value_cols if col not in [year_col, quarter_col]]
    
    # Maps quarter to month and end date
    quarter_to_month_day = {"Q1": (3, 31), "Q2": (6, 30), "Q3": (9, 30), "Q4": (12, 31)}
    
    # Create column QuarterEnd
    df["QuarterEnd"] = df.apply(
        lambda row: pd.Timestamp(year=int(row[year_col]), 
                                 month=quarter_to_month_day[row[quarter_col]][0],
                                 day=quarter_to_month_day[row[quarter_col]][1]),
        axis=1
    )
    
    df = df.set_index("QuarterEnd")
    
    # sort index in chronological order (required for resample)
    df = df.sort_index()
    
    # quarterly resample
    df_quarterly = df[value_cols].resample("Q").mean().reset_index()
    
    # order descending by date for output
    df_quarterly = df_quarterly.sort_values("QuarterEnd", ascending=False)
    
    df_quarterly.to_csv(f"quarterly_data/{filename}_quarterly.csv", index=False)


In [51]:
df_diesel = pd.read_csv("monthly_data/EIA_Diesel_Price_Monthly.csv", skiprows=4)
df_diesel.head()

,Month,U.S. No 2 Diesel Ultra Low Sulfur (0-15 ppm) Retail Prices Dollars per Gallon
0,Oct 2025,3.679
1,Sep 2025,3.748
2,Aug 2025,3.744
3,Jul 2025,3.779
4,Jun 2025,3.599


In [52]:
normalize_to_quarterly(df_diesel, "Month", "U.S. No 2 Diesel Ultra Low Sulfur (0-15 ppm) Retail Prices Dollars per Gallon", "EIA_Diesel_Price")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df[date_col])
/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [53]:
df_policy_unc = pd.read_csv("monthly_data/US_Policy_Uncertainty.csv")
df_policy_unc.drop(df_policy_unc.index[-1], inplace=True)
df_policy_unc.head()

,Year,Month,News_Based_Policy_Uncert_Index
0,2025,10.0,282.421417
1,2025,9.0,321.231079
2,2025,8.0,370.925415
3,2025,7.0,372.062683
4,2025,6.0,375.585297


In [54]:
quarterly_from_year_month(df_policy_unc, "Year", "Month", "News_Based_Policy_Uncert_Index", "US_Policy_Uncertainty")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/488090177.py:23: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  quarterly = df[[value_col]].resample("Q").mean().reset_index()


In [55]:
df_electricity = pd.read_csv("monthly_data/EIA_Electricity_Price_Monthly.csv", skiprows=4)
df_electricity.head()

,Month,all sectors cents per kilowatthour,residential cents per kilowatthour,commercial cents per kilowatthour,industrial cents per kilowatthour
0,Jul 2025,14.26,17.62,14.04,9.06
1,Jul 2025,NaN,NaN,NaN,NaN
2,Jul 2025,NaN,NaN,NaN,NaN
3,Jun 2025,14.39,17.47,14.15,9.31
4,May 2025,13.88,17.47,13.62,8.87


In [56]:
normalize_to_quarterly(
    df_electricity,
    date_col="Month",
    value_cols=[
        "all sectors cents per kilowatthour",
        "residential cents per kilowatthour",
        "commercial cents per kilowatthour",
        "industrial cents per kilowatthour"
    ],
    filename="EIA_Electricity_Price"
)

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df[date_col])
/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [57]:
df_naturalgas = pd.read_csv("monthly_data/EIA_Natural_Gas_Price_Monthly.csv", skiprows=4)
df_naturalgas.head()


,Month,Henry Hub Natural Gas Spot Price Dollars per Million Btu
0,Sep 2025,2.97
1,Aug 2025,2.91
2,Jul 2025,3.20
3,Jun 2025,3.02
4,May 2025,3.12


In [58]:
normalize_to_quarterly(df_naturalgas, "Month", "Henry Hub Natural Gas Spot Price Dollars per Million Btu", "EIA_Natural_Gas_Price")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df[date_col])
/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [59]:
df_graphite = pd.read_csv("monthly_data/Graphite_Electrode_Price_monthly.csv")
df_graphite.head()

,observation_date,PCU3359913359910
0,1960-12-01,25.6
1,1961-01-01,25.6
2,1961-02-01,25.6
3,1961-03-01,25.6
4,1961-04-01,25.6


In [60]:
normalize_to_quarterly(df_graphite, "observation_date", "PCU3359913359910", "Graphite_Electrode_Price")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [61]:
df_railprice = pd.read_csv("monthly_data/BLS_rail_price_monthly.csv")
df_railprice.head()

,observation_date,PCU48214821
0,1996-12-01,100.0
1,1997-01-01,99.9
2,1997-02-01,100.0
3,1997-03-01,100.1
4,1997-04-01,100.1


In [62]:
normalize_to_quarterly(df_railprice, "observation_date", "PCU48214821", "BLS_rail_price")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [63]:
df_ferrousmetal = pd.read_csv("monthly_data/PPI_Ferrous_Metal_Scrap.csv")
df_ferrousmetal.head()

,observation_date,PCU33123312
0,2003-12-01,100.0
1,2004-01-01,102.8
2,2004-02-01,108.8
3,2004-03-01,114.5
4,2004-04-01,125.3


In [64]:
normalize_to_quarterly(df_ferrousmetal, "observation_date", "PCU33123312", "PPI_Ferrous_Metal_Scrap")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [65]:
df_iron = pd.read_csv("monthly_data/PPI_Iron_Steel_Scrap_Monthly.csv")
df_iron.head()

,observation_date,WPS101
0,1967-01-01,29.3
1,1967-02-01,29.3
2,1967-03-01,29.4
3,1967-04-01,29.5
4,1967-05-01,29.4


In [66]:
normalize_to_quarterly(df_iron, "observation_date", "WPS101", "PPI_Iron_Steel_Scrap")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [67]:
df_wage = pd.read_csv("monthly_data/hourly_wage_monthly.csv")
df_wage.head()

,observation_date,CES0500000003
0,2006-03-01,20.05
1,2006-04-01,20.15
2,2006-05-01,20.13
3,2006-06-01,20.23
4,2006-07-01,20.29


In [68]:
normalize_to_quarterly(df_wage, "observation_date", "CES0500000003", "Hourly_Wage")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [69]:
df_commodities = pd.read_csv("monthly_data/PPI_All_Commodities.csv")
df_commodities.head()

,observation_date,PPIACO
0,1913-01-01,12.1
1,1913-02-01,12.0
2,1913-03-01,12.0
3,1913-04-01,12.0
4,1913-05-01,11.9


In [70]:
normalize_to_quarterly(df_commodities, "observation_date", "PPIACO", "PPI_All_Commodities")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [71]:
df_dollar = pd.read_csv("monthly_data/US_Dollar_Index.csv")
df_dollar.head()

,observation_date,DTWEXBGS
0,2015-11-09,111.9112
1,2015-11-10,111.9919
2,2015-11-11,NaN
3,2015-11-12,111.7965
4,2015-11-13,112.0264


In [72]:
normalize_daily_to_quarterly(
    df_dollar,
    date_col="observation_date",
    value_cols="DTWEXBGS",
    filename="US_Dollar_Index"
)

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/2602947297.py:23: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.set_index("Date")[available_cols]
/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/2602947297.py:29: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df_quarterly = df_monthly.resample("Q").mean().reset_index()


In [73]:
df_disasters = pd.read_csv("monthly_data/NOAA_disasters_clean.csv")
df_disasters.head()

,Name,Disaster,Begin Date,End Date,CPI-Adjusted Cost,Unadjusted Cost,Deaths
0,Southeast/Ohio Valley Severe Weather (February...,Severe Storm,2009-02-10,2009-02-11,2592.9,1740.2,10
1,Midwest/Southeast Tornadoes (March 2009),Severe Storm,2009-03-25,2009-03-28,2427.9,1640.5,0
2,South/Southeast Severe Weather and Tornadoes (...,Severe Storm,2009-04-09,2009-04-10,2116.4,1430.1,6
3,Central Derecho and Tornadoes (May 2009),Severe Storm,2009-05-07,2009-05-09,1274.2,861.0,7
4,"Midwest, South and East Severe Weather (June 2...",Severe Storm,2009-06-09,2009-06-12,1939.0,1328.1,0


In [74]:
disasters_to_quarterly(
    df_disasters,
    begin_col="Begin Date",
    cost_cols=["CPI-Adjusted Cost", "Unadjusted Cost"],
    filename="NOAA_disasters"
)

In [3]:
#extracting dataset for feature X8 from US Census API


import requests

base_url = "https://api.census.gov/data/timeseries/intltrade/exports/hs"
years = range(2015, 2026)  # 2015-2024

all_data = []

for year in years:
    params = {
        "get": "YEAR,MONTH,E_COMMODITY,ALL_VAL_MO",
        "time": str(year)
    }
    response = requests.get(base_url, params=params)
    if response.status_code != 200:
        raise Exception(f"Errore API {year}: {response.status_code} — {response.text}")
    
    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    all_data.append(df)

# Concatenazione finale
df_all = pd.concat(all_data, ignore_index=True)

# Conversioni numeriche
df_all["ALL_VAL_MO"] = pd.to_numeric(df_all["ALL_VAL_MO"], errors="coerce")
df_all["MONTH"] = pd.to_numeric(df_all["MONTH"], errors="coerce")

# Colonna datetime
df_all["date"] = pd.to_datetime(df_all["YEAR"] + "-" + df_all["MONTH"].astype(int).astype(str) + "-01")

df_all.head()

df_all.to_csv("monthly_data/US_Scrap_Exports.csv", index=False)


In [4]:
df_exports = pd.read_csv("monthly_data/US_Scrap_Exports.csv")
df_exports.head()

,YEAR,MONTH,E_COMMODITY,ALL_VAL_MO,time,date
0,2015,1,600544,4546,2015-01,2015-01-01
1,2015,1,6005420000,53239,2015-01,2015-01-01
2,2015,1,600542,53239,2015-01,2015-01-01
3,2015,1,6005410000,8175,2015-01,2015-01-01
4,2015,1,600541,8175,2015-01,2015-01-01


In [5]:

#from us scrap exports dataset, extract only scrap related to metal materials

df_exports = pd.read_csv("monthly_data/US_Scrap_Exports.csv")

df_exports["ALL_VAL_MO"] = pd.to_numeric(df_exports["ALL_VAL_MO"], errors="coerce")

df_exports["date"] = pd.to_datetime(df_exports["YEAR"].astype(str) + "-" + df_exports["MONTH"].astype(str) + "-01")

# HS codes for scrap
hs_scrap_prefixes = ["7204", "7404", "7602", "7802", "7902"]

# rows filtering for scrap codes
df_scrap = df_exports[df_exports["E_COMMODITY"].astype(str).str[:4].isin(hs_scrap_prefixes)]

# group by month: sum values
monthly_scrap = df_scrap.groupby("date").agg(
    US_Scrap_Exports=("ALL_VAL_MO", "sum")
).sort_index()

monthly_scrap.to_csv("monthly_data/US_Scrap_Exports_monthly.csv")

print(monthly_scrap.head(12))  


            US_Scrap_Exports
date                        
2015-01-01        2226209286
2015-02-01        2119999047
2015-03-01        2429985030
2015-04-01        2558674800
2015-05-01        2838245400
2015-06-01        2761367685
2015-07-01        2388130581
2015-08-01        2294241798
2015-09-01        2242218408
2015-10-01        2104812585
2015-11-01        1985854725
2015-12-01        1866635025


In [6]:
df_exports_monthly = pd.read_csv("monthly_data/US_Scrap_Exports_monthly.csv")
df_exports_monthly.head()

,date,US_Scrap_Exports
0,2015-01-01,2226209286
1,2015-02-01,2119999047
2,2015-03-01,2429985030
3,2015-04-01,2558674800
4,2015-05-01,2838245400


In [8]:
normalize_to_quarterly(df_exports_monthly, "date", "US_Scrap_Exports", "US_Scrap_Exports")

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1509/3612523773.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df.set_index("Date")[available_cols]


In [84]:
df_nucor_incomestatement = pd.read_csv("monthly_data/nucor_02_INCOME_STATEMENT.csv")
df_nucor_incomestatement.head()

,AccessionNumber,PeriodEndDate,FilingDate,Year,Quarter,Net_sales_millions,COGS_millions,Tons_Shipped,Tons_Pct_Change
0,0001193125-25-276207,2025-10-04,2025-11-12,2025,Q3,8521.000,7333.000,6774000.0,9.0
1,0000950170-25-107646,2025-07-05,2025-08-13,2025,Q2,8456.000,7233.000,6820000.0,8.0
2,0000950170-25-071567,2025-04-05,2025-05-14,2025,Q1,7830.000,7225.000,6830000.0,10.0
3,0000950170-24-121863,2024-09-28,2024-11-06,2024,Q3,7444.160,6686.226,6196000.0,-1.0
4,0000950170-24-092479,2024-06-29,2024-08-07,2024,Q2,8077.172,6883.117,6289000.0,-5.0


In [85]:
resample_quarterly_nucor(
    df_nucor_incomestatement,
    year_col="Year",
    quarter_col="Quarter",
    value_cols=["Net_sales_millions", "COGS_millions", "Tons_Shipped"],
    filename = "nucor_02_INCOME_STATEMENT"
)

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/1019361967.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df_quarterly = df[value_cols].resample("Q").mean().reset_index()


In [86]:
df_nucor_metrics = pd.read_csv("monthly_data/nucor_04_METRICS.csv")
df_nucor_metrics.head()

,Year,Quarter,PeriodEndDate,Cost_Per_Ton,Gross_Margin_Pct,Inventories_millions,Inventories_Previous,Average_Inventory,Inventory_Turnover,Days_Inventory_Outstanding
0,2025,Q3,2025-10-04,1082.521405,13.94,5393.000,5126.493,5259.7465,1.3942,64.6
1,2025,Q2,2025-07-05,1060.557185,14.46,5462.000,5255.843,5358.9215,1.3497,66.7
2,2025,Q1,2025-04-05,1057.833089,7.73,5256.000,5589.675,5422.8375,1.3323,67.6
3,2024,Q3,2024-09-28,1079.119755,10.18,5126.493,5246.365,5186.4290,1.2892,69.8
4,2024,Q2,2024-06-29,1094.469232,14.78,5255.843,5632.324,5444.0835,1.2643,71.2


In [87]:
resample_quarterly_nucor(
    df_nucor_metrics,
    year_col="Year",
    quarter_col="Quarter",
    value_cols=["Cost_Per_Ton", "Gross_Margin_Pct", "Inventories_millions", "Inventories_Previous", "Average_Inventory", "Inventory_Turnover", "Days_Inventory_Outstanding"],
    filename = "nucor_04_METRICS"
)

/var/folders/z_/h8pm6bx50mngk0pbk5p07btc0000gn/T/ipykernel_1317/1019361967.py:26: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  df_quarterly = df[value_cols].resample("Q").mean().reset_index()
